In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary
import matplotlib.pyplot as plt
from torchvision.datasets import ImageFolder
from torchvision import transforms
import torchvision.models as models
from PIL import Image
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
from sklearn.metrics import accuracy_score, f1_score
import torch.optim.lr_scheduler as lr_scheduler

from torch.utils.data import random_split

In [2]:
import os
import cv2
import numpy as np

# 텍스트 파일 경로
annotation_file_train = '/mnt/e/Blur_Dot/MLT/coords_train_230000.txt'
image_dir_train = '/mnt/e/Blur_Dot/MLT/ch8_training_word_images_gt_part_2'

# 데이터 저장 리스트
image_paths_train = []
bounding_boxes_train = []

# 파일 읽기
with open(annotation_file_train, "r") as f:
    lines = f.readlines()
    for line in lines:
        parts = line.strip().split(",")
        image_name = parts[0]
        coords = list(map(int, parts[1:]))  # 좌표를 정수로 변환
        bbox = np.array(coords).reshape(-1, 2)  # (x, y) 좌표 쌍 생성
        image_paths_train.append(os.path.join(image_dir_train, image_name))
        bounding_boxes_train.append(bbox)

In [3]:
import os
import cv2
import numpy as np

# 텍스트 파일 경로
annotation_file_val = '/mnt/e/Blur_Dot/MLT/coords_val_5000.txt'
image_dir_val = '/mnt/e/Blur_Dot/MLT/ch8_validation_word_images_gt'

# 데이터 저장 리스트
image_paths_val = []
bounding_boxes_val = []

# 파일 읽기
with open(annotation_file_val, "r") as f:
    lines = f.readlines()
    for line in lines:
        parts = line.strip().split(",")
        image_name = parts[0]
        coords = list(map(int, parts[1:]))  # 좌표를 정수로 변환
        bbox = np.array(coords).reshape(-1, 2)  # (x, y) 좌표 쌍 생성
        image_paths_val.append(os.path.join(image_dir_val, image_name))
        bounding_boxes_val.append(bbox)

In [4]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
transform = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.RandomResizedCrop(224),
        transforms.ToTensor(),
        transforms.ConvertImageDtype(torch.float),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

In [6]:
class TextDetectionDataset(Dataset):
    def __init__(self, image_paths, bounding_boxes, transform=None):
        self.image_paths = image_paths
        self.bounding_boxes = bounding_boxes
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # 이미지 경로 확인
        image_path = self.image_paths[idx]
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image not found: {image_path}")

        # 이미지 로드
        image = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"Failed to load image: {image_path}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # BGR to RGB 변환
        
        # 이미지를 PIL Image로 변환
        image = Image.fromarray(image)
        
        bbox = self.bounding_boxes[idx]

        # Transform 적용
        if self.transform:
            image = self.transform(image)

        return image, bbox

# 데이터셋 생성
dataset = TextDetectionDataset(image_paths_train, bounding_boxes_train, transform=transform)


In [7]:
from craft import CRAFT

# 모델 로드
model = CRAFT(pretrained=True)
state_dict = torch.load('/home/songeun/LogoDetection/CRAFT_B/CRAFT-pytorch-master/weights/craft_ic15_20k.pth', map_location=torch.device('cpu'))
new_state_dict = {}
for k, v in state_dict.items():
    new_key = k.replace('module.', '')  # Remove 'module.' prefix
    new_state_dict[new_key] = v

model.load_state_dict(new_state_dict)

model.train()

# GPU 사용 설정
model = model.to(DEVICE)

print(model)


CRAFT(
  (basenet): vgg16_bn(
    (slice1): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
      (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (7): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (9): ReLU(inplace=True)
      (10): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (slice2): Sequential(
      (12): ReLU(inplace=True)
      (13): MaxPool2d(kerne

In [8]:
import torch.nn.functional as F

def craft_loss(y_pred, y_true):
    # y_pred와 y_true의 크기를 확인하여 마지막 채널만을 사용하는 MSE 계산
    # 예측값의 채널별 손실 계산
    text_loss = F.mse_loss(y_pred[:, :, :, 0], y_true[:, :, :, 0])
    link_loss = F.mse_loss(y_pred[:, :, :, 1], y_true[:, :, :, 1])
    return text_loss + link_loss


In [9]:
# 데이터로더 생성
dataset_train = TextDetectionDataset(image_paths_train, bounding_boxes_train, transform=transform)
dataset_val = TextDetectionDataset(image_paths_val, bounding_boxes_val, transform=transform)
train_dl = DataLoader(dataset_train, batch_size=32, shuffle=True)
val_dl = DataLoader(dataset_val, batch_size=32, shuffle=True)

In [10]:
summary(model)

Layer (type:depth-idx)                   Param #
CRAFT                                    --
├─vgg16_bn: 1-1                          --
│    └─Sequential: 2-1                   --
│    │    └─Conv2d: 3-1                  1,792
│    │    └─BatchNorm2d: 3-2             128
│    │    └─ReLU: 3-3                    --
│    │    └─Conv2d: 3-4                  36,928
│    │    └─BatchNorm2d: 3-5             128
│    │    └─ReLU: 3-6                    --
│    │    └─MaxPool2d: 3-7               --
│    │    └─Conv2d: 3-8                  73,856
│    │    └─BatchNorm2d: 3-9             256
│    │    └─ReLU: 3-10                   --
│    │    └─Conv2d: 3-11                 147,584
│    │    └─BatchNorm2d: 3-12            256
│    └─Sequential: 2-2                   --
│    │    └─ReLU: 3-13                   --
│    │    └─MaxPool2d: 3-14              --
│    │    └─Conv2d: 3-15                 295,168
│    │    └─BatchNorm2d: 3-16            512
│    │    └─ReLU: 3-17                   --
│

In [11]:

for named, param in model.named_parameters():
    print(f"[{named}] - {param.shape}")

    param.requires_grad = False

[basenet.slice1.0.weight] - torch.Size([64, 3, 3, 3])
[basenet.slice1.0.bias] - torch.Size([64])
[basenet.slice1.1.weight] - torch.Size([64])
[basenet.slice1.1.bias] - torch.Size([64])
[basenet.slice1.3.weight] - torch.Size([64, 64, 3, 3])
[basenet.slice1.3.bias] - torch.Size([64])
[basenet.slice1.4.weight] - torch.Size([64])
[basenet.slice1.4.bias] - torch.Size([64])
[basenet.slice1.7.weight] - torch.Size([128, 64, 3, 3])
[basenet.slice1.7.bias] - torch.Size([128])
[basenet.slice1.8.weight] - torch.Size([128])
[basenet.slice1.8.bias] - torch.Size([128])
[basenet.slice1.10.weight] - torch.Size([128, 128, 3, 3])
[basenet.slice1.10.bias] - torch.Size([128])
[basenet.slice1.11.weight] - torch.Size([128])
[basenet.slice1.11.bias] - torch.Size([128])
[basenet.slice2.14.weight] - torch.Size([256, 128, 3, 3])
[basenet.slice2.14.bias] - torch.Size([256])
[basenet.slice2.15.weight] - torch.Size([256])
[basenet.slice2.15.bias] - torch.Size([256])
[basenet.slice2.17.weight] - torch.Size([256, 256

In [12]:
print(model)

CRAFT(
  (basenet): vgg16_bn(
    (slice1): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
      (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (7): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (9): ReLU(inplace=True)
      (10): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (slice2): Sequential(
      (12): ReLU(inplace=True)
      (13): MaxPool2d(kerne

In [13]:
# model.classifier = nn.Sequential(
#     nn.Linear(in_features=1280, out_features=900),
#     nn.Linear(in_features=900, out_features=600),
#     nn.Linear(in_features=600, out_features=300),
#     nn.Linear(300, 1),
# )

In [14]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=10, verbose=True
)

In [15]:

# # 사전학습된 모델의 파라미터 비활성화 설정
# for named, param in model.classifier.named_parameters():
#     print(f"[{named}] - {param.shape}")
#     # 역전파 시에 업데이트 되지 않도록 설정
#     param.requires_grad = False

In [16]:
# for named, param in model.classifier.named_parameters():
#     print(f"[{named}] - {param.shape}")
#     # 역전파 시에 업데이트 되도록 설정
#     param.requires_grad = True

In [17]:

## models 폴더 아래 프로젝트 폴더 아래 모델 파일저장
import os

# 저장 경로
SAVE_PATH = "/home/songeun/LogoDetection/CRAFT_B/CRAFT-pytorch-master/models/"
# 저장 파일명
SAVE_FILE = "top_model_train_wbs.pth"
# 저장 모델구조 및 파라미터 모두 저장
SAVE_MODEL = "top_model_all.pth"

if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

In [18]:
for feature, target in train_dl:
    print(feature.shape, target.shape)
    break

torch.Size([32, 3, 224, 224]) torch.Size([32, 4, 2])


In [19]:
def resize_target(target, output_size):
    target = target.float()
    batch_size = target.size(0)
    
    # 0으로 초기화된 텐서 생성
    target_resized = torch.zeros(batch_size, output_size[0], output_size[1], 2, device=target.device)
    
    for b in range(batch_size):
        # 좌표 스케일링 시 안전한 처리
        scaled_coords = target[b].clone()
        
        # 좌표가 [0, 1] 범위에 있다고 가정
        scaled_x = scaled_coords[:, 0] * (output_size[1] - 1)  # x 좌표
        scaled_y = scaled_coords[:, 1] * (output_size[0] - 1)  # y 좌표
        
        # 정수 인덱스로 안전하게 변환
        scaled_x = torch.clamp(scaled_x.round().long(), 0, output_size[1] - 1)
        scaled_y = torch.clamp(scaled_y.round().long(), 0, output_size[0] - 1)
        
        # 해당 위치에 1로 표시
        for x, y in zip(scaled_x, scaled_y):
            target_resized[b, y, x, :] = 1.0
    
    return target_resized

In [20]:
# annotations 차원 변환 함수
def adjust_annotations(annotations):
    if annotations.dim() == 3:
        annotations = annotations.unsqueeze(-1)  # 채널 차원 추가 (예: [batch_size, height, width, 1])
        annotations = annotations.repeat(1, 1, 1, 2)  # 채널을 2로 반복하여 [batch_size, height, width, 2] 형태로 만듦
    return annotations

In [ ]:
def calculate_binary_metrics(y_true, y_pred, threshold=0.5):
    # 이진 분류를 위한 임계값 적용
    y_pred_binary = (y_pred > threshold).astype(int)
    y_true_binary = (y_true > threshold).astype(int)
    
    accuracy = accuracy_score(y_true_binary, y_pred_binary)
    f1 = f1_score(y_true_binary, y_pred_binary)
    
    return accuracy, f1


: 

In [ ]:
model.to(DEVICE)
model.train()

epochs = 1000
loss_history = []
f1score_history = []
accuracy_history = []

for epoch in range(epochs):
    
    epoch_loss = 0.0
    train_pred = []
    train_true = []

    for images, annotations in train_dl:
        images = images.to(DEVICE)
        images.requires_grad = True
        annotations = annotations.to(DEVICE)

        # print("Images requires_grad:", images.requires_grad)  # True 확인 필요
        # print("Annotations requires_grad:", annotations.requires_grad)  # False여도 무방

        # 순전파
        optimizer.zero_grad()
        y_pred, _ = model(images)

        # annotations 차원 맞추기
        annotations = adjust_annotations(annotations)

        # 타겟 크기 확장
        annotations_resized = resize_target(annotations, (y_pred.shape[1], y_pred.shape[2]))
        
        # 손실 계산
        loss = craft_loss(y_pred, annotations_resized)
        
        # 역전파 및 최적화
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        # 현재 배치의 예측값과 타겟값 추가
        train_pred.append(y_pred.detach().cpu())
        train_true.append(annotations_resized.detach().cpu())

    # 배치별 예측값과 타겟값 병합
    train_pred = torch.cat(train_pred).numpy().flatten()
    train_true = torch.cat(train_true).numpy().flatten()

    train_accuracy, train_f1 = calculate_binary_metrics(train_true, train_pred)
    avg_loss_train = epoch_loss / len(train_dl)

    loss_history.append(avg_loss_train)
    f1score_history.append(train_f1)
    accuracy_history.append(train_accuracy)
    
    print(f"EPOCH [{epoch+1}/{epochs}]")
    print(f"[TRAIN] Loss: {avg_loss_train}, Score: {train_f1}")

    # 최적화 스케줄러
    scheduler.step(loss)
    
    print()
    print(f"scheduler.num_bad_epochs: {scheduler.num_bad_epochs}", end=" ")
    # PyTorch에서 학습률 스케줄러(Scheduler)를 사용할 때, 현재 학습률이 개선되지 않은(epoch의 손실이 향상되지 않은) 연속적인 epoch의 수를 나타내는 변수
    print(f"scheduler.patience: {scheduler.patience}")
    print()

    if len(f1score_history) == 1:

        # 첫번째라서 무조건 모델 파라미터 저장
        torch.save(model.state_dict(), SAVE_PATH + SAVE_FILE)

        # 모델 전체 저장
        torch.save(model, SAVE_PATH + SAVE_MODEL)

    else:
        if f1score_history[-1] >= max(f1score_history):
            torch.save(model.state_dict(), SAVE_PATH + SAVE_FILE)
            # 모델 전체 저장
            torch.save(model, SAVE_PATH + SAVE_MODEL)

    # 손실 감소(성능 개선) 안 되는 경우 조기 종료
    if scheduler.num_bad_epochs >= scheduler.patience:
        print()
        print(f"{scheduler.patience} EPOCH 성능 개선 없어서 조기 종료")
        break

# 테스트
model.eval()
val_pred = []
val_true = []

with torch.no_grad():
    for features, targets in val_dl:

        features = features.to(DEVICE)
        targets = targets.to(DEVICE)

        outputs, _ = model(features.float())

        val_pred.append(outputs.cpu())
        val_true.append(targets.cpu())

val_pred = torch.cat(val_pred).numpy().flatten()
val_true = torch.cat(val_true).numpy().flatten()
        
val_accuracy, val_f1 = calculate_binary_metrics(val_true, val_pred)


print(f"Accuracy: {val_accuracy:.4f}")
print(f"F1 Score: {val_f1:.4f}")
